In [ ]:
# Cell 1: Install Dependencies
!pip install GDAL

In [ ]:
# Cell 2: Core Algorithm Classes (Clean Version)
import numpy as np
from numpy import ma
from osgeo import gdal, osr
import scipy.ndimage as ndi # Importado una sola vez al principio

# ==============================================================================
# 1. UTILITIES AND DATA PROVIDER (dataprovider.py & utils.py)
# ==============================================================================
def get_gradations(arr):
    return np.unique(arr).tolist()

class Raster:
    """Clase adaptada para manejar datos ráster."""
    def __init__(self, filename=None):
        self.filename = filename
        self.bands = None
        self.bandcount = 0
        self.geodata = {}
        if filename: self._read()

    def geoDataMatch(self, other_raster):
        """Compara si la geometría de dos rásters coincide."""
        geodata1 = self.getGeodata()
        geodata2 = other_raster.getGeodata()
        if geodata1.get('xSize') != geodata2.get('xSize') or geodata1.get('ySize') != geodata2.get('ySize'):
            print(f"ERROR: Las dimensiones no coinciden. {self.filename.split('/')[-1]} ({geodata1.get('xSize')}x{geodata1.get('ySize')}) vs {other_raster.filename.split('/')[-1]} ({geodata2.get('xSize')}x{geodata2.get('ySize')})")
            return False
        t1 = geodata1.get('transform')
        t2 = geodata2.get('transform')
        if not np.allclose(t1, t2, atol=1e-6):
            print(f"ERROR: La geotransformación no coincide. {self.filename.split('/')[-1]} vs {other_raster.filename.split('/')[-1]}")
            return False
        return True

    def _read(self):
        data = gdal.Open(self.filename)
        self.geodata['xSize'], self.geodata['ySize'] = data.RasterXSize, data.RasterYSize
        self.geodata['proj'], self.geodata['transform'] = data.GetProjection(), data.GetGeoTransform()
        self.bandcount = data.RasterCount
        self.bands = []
        for i in range(1, self.bandcount + 1):
            band = data.GetRasterBand(i)
            nodata = band.GetNoDataValue()
            arr = band.ReadAsArray()
            mask = (arr == nodata) if nodata is not None else np.zeros_like(arr, dtype=bool)
            self.bands.append(ma.array(arr, mask=mask))

    def create(self, bands_list, geodata):
        self.bands = [ma.array(b) for b in bands_list]
        self.bandcount = len(self.bands)
        self.geodata = geodata

    def getBand(self, bandNo): return self.bands[bandNo - 1]
    def getBandGradation(self, bandNo): return get_gradations(self.getBand(bandNo).compressed())
    def getGeodata(self): return self.geodata
    def getNeighbourhoodSize(self, ns): return (2 * ns + 1) ** 2

    def getNeighbours(self, row, col, ns, bandNo=1):
        band = self.getBand(bandNo)
        return band[row - ns : row + ns + 1, col - ns : col + ns + 1]

    def normalize(self, mode='mean'):
        for i in range(self.bandcount):
            band = self.bands[i]
            if mode == 'mean':
                mean, std = band.mean(), band.std()
                if std > 0: self.bands[i] = (band - mean) / std
            elif mode == 'maxmin':
                min_val, max_val = band.min(), band.max()
                if (max_val - min_val) > 0: self.bands[i] = (band - min_val) / (max_val - min_val)

    def save(self, filename):
        driver = gdal.GetDriverByName('GTiff')
        out_ds = driver.Create(filename, self.geodata['xSize'], self.geodata['ySize'], self.bandcount, gdal.GDT_Int16) # Cambiado a Int16 para aceptar -9999
        out_ds.SetProjection(self.geodata['proj'])
        out_ds.SetGeoTransform(self.geodata['transform'])
        for i in range(self.bandcount):
            out_band = out_ds.GetRasterBand(i + 1)
            out_band.WriteArray(self.bands[i].filled(-9999))
            out_band.SetNoDataValue(-9999)
        out_ds.FlushCache()
        out_ds = None

# ==============================================================================
# 2. CÁLCULO DE MATRIZ DE TRANSICIÓN (crosstabs/model.py)
# ==============================================================================
class CrossTable:
    def __init__(self, band1, band2):
        mask = band1.mask | band2.mask
        band1.mask, band2.mask = mask, mask
        self.X, self.Y = band1.compressed(), band2.compressed()
        self.graduation_x, self.graduation_y = get_gradations(self.X), get_gradations(self.Y)
        self.rows, self.cols = len(self.graduation_x), len(self.graduation_y)
        self._T = None

    def computeCrosstable(self):
        self._T = np.zeros([self.rows, self.cols], dtype=int)
        for i in range(len(self.X)):
            class_num_x = self.graduation_x.index(self.X[i])
            class_num_y = self.graduation_y.index(self.Y[i])
            self._T[class_num_x][class_num_y] += 1

    def getCrosstable(self):
        if self._T is None: self.computeCrosstable()
        return self._T

    def getTransition(self, fromClass, toClass):
        i, j = self.graduation_x.index(fromClass), self.graduation_y.index(toClass)
        return self.getCrosstable()[i, j]

# ==============================================================================
# 3. MUESTREADOR DE DATOS (sampler/sampler.py)
# ==============================================================================
class Sampler:
    def __init__(self, state_raster, factor_rasters, ns=1):
        self.state, self.factors, self.ns = state_raster, factor_rasters, ns
        self.data = None

    def _getSample(self, output_raster, row, col):
        # Descomenta los prints si necesitas depurar de nuevo
        # print(f"\n--- Intentando crear muestra para el píxel en (Fila: {row}, Col: {col}) ---")

        factors_neighbors = []
        for i, factor in enumerate(self.factors):
            neighbors = factor.getNeighbours(row, col, self.ns)
            if neighbors.mask.any():
                # print(f"    >>> Muestra descartada: NoData en Factor {i+1}.")
                return None
            factors_neighbors.append(neighbors.flatten())

        state_neighbors = self.state.getNeighbours(row, col, self.ns)
        if state_neighbors.mask.any():
            # print(f"    >>> Muestra descartada: NoData en Estado Inicial.")
            return None

        output_val = output_raster.getBand(1)[row, col]
        if ma.is_masked(output_val):
            return None

        # print("    +++ MUESTRA CREADA EXITOSAMENTE +++")
        return {'state': state_neighbors.flatten(), 'factors': np.hstack(factors_neighbors), 'output': output_val}

    def setTrainingData(self, output_raster, mode='Stratified', samples=5000):
        rows, cols = self.state.geodata['ySize'], self.state.geodata['xSize']
        collected_samples = []

        if mode == 'Random':
            while len(collected_samples) < samples:
                r = np.random.randint(self.ns, rows - self.ns)
                c = np.random.randint(self.ns, cols - self.ns)
                sample = self._getSample(output_raster, r, c)
                if sample:
                    collected_samples.append(sample)

        elif mode == 'Stratified':
            categories = output_raster.getBandGradation(1)
            band = output_raster.getBand(1)
            if not categories:
                 self.data = []
                 return
            samples_per_cat = int(samples / len(categories))

            for cat in categories:
                valid_pixels_mask = ~band.mask
                rows_cat, cols_cat = np.where((band.data == cat) & valid_pixels_mask)
                cat_indices = list(zip(rows_cat, cols_cat))

                if not cat_indices:
                    print(f"Advertencia: No se encontraron píxeles válidos para la Clase {cat} en el mapa final.")
                    continue

                count = 0
                while count < samples_per_cat:
                    if not cat_indices: break
                    idx = np.random.randint(len(cat_indices))
                    r, c = cat_indices.pop(idx)
                    if self.ns <= r < rows - self.ns and self.ns <= c < cols - self.ns:
                        sample = self._getSample(output_raster, r, c)
                        if sample:
                            collected_samples.append(sample)
                            count += 1

        np.random.shuffle(collected_samples)
        self.data = collected_samples

# ==============================================================================
# 4. PERCEPTRÓN MULTICAPA (mlp/model.py)
# ==============================================================================
class MLP:
    def __init__(self, *args):
        self.shape = args
        self.layers = [np.ones(s + 1) if i == 0 else np.ones(s) for i, s in enumerate(self.shape)]
        self.weights = [np.zeros((self.layers[i].size, self.layers[i + 1].size)) for i in range(len(self.shape) - 1)]
        self.dw = [0] * len(self.weights)
        self.reset()

    def reset(self):
        for i in range(len(self.weights)):
            Z = np.random.random(self.weights[i].shape)
            self.weights[i][...] = (2 * Z - 1) * 0.25

    def propagate_forward(self, data):
        self.layers[0][0:-1] = data
        for i in range(1, len(self.shape)):
            self.layers[i][...] = np.tanh(np.dot(self.layers[i - 1], self.weights[i - 1]))
        return self.layers[-1]

    def propagate_backward(self, target, lrate=0.01, momentum=0.01):
        deltas, error = [], target - self.layers[-1]
        deltas.append((1.0 - self.layers[-1]**2) * error)
        for i in range(len(self.shape) - 2, 0, -1):
            deltas.insert(0, np.dot(deltas[0], self.weights[i].T) * (1.0 - self.layers[i]**2))
        for i in range(len(self.weights)):
            layer, delta = np.atleast_2d(self.layers[i]), np.atleast_2d(deltas[i])
            dw = np.dot(layer.T, delta)
            self.weights[i] += lrate * dw + momentum * self.dw[i]
            self.dw[i] = dw

# ==============================================================================
# 5. SIMULADOR UNIFICADO (mlp/manager.py & simulator/sim.py)
# ==============================================================================
class MlpSimulator:
    def __init__(self, initial_raster, factor_rasters, mlp_model, crosstable, ns=1):
        self.initial_raster, self.factor_rasters = initial_raster, factor_rasters
        self.mlp, self.crosstable, self.ns = mlp_model, crosstable, ns

    def _get_prediction_input_vector(self, row, col):
        state_neighbors = self.initial_raster.getNeighbours(row, col, self.ns)
        if state_neighbors.mask.any(): return None
        state_neighbors_flat = state_neighbors.flatten()
        factors_neighbors = []
        for factor in self.factor_rasters:
            neighbors = factor.getNeighbours(row, col, self.ns)
            if neighbors.mask.any(): return None
            factors_neighbors.append(neighbors.flatten())
        state_categories = self.crosstable.graduation_x
        num_state_neurons_per_pix = len(state_categories) - 1
        dummy_vector = np.array([])
        for pixel_state in state_neighbors_flat:
            state_dummy = np.zeros(num_state_neurons_per_pix)
            if pixel_state != state_categories[-1]:
                 idx = state_categories.index(pixel_state)
                 state_dummy[idx] = 1.0
            dummy_vector = np.hstack([dummy_vector, state_dummy])
        return np.hstack([dummy_vector, np.hstack(factors_neighbors)])

    def run_simulation(self):
        rows, cols = self.initial_raster.geodata['ySize'], self.initial_raster.geodata['xSize']
        potential_maps = {cat: np.zeros((rows, cols)) for cat in self.crosstable.graduation_y}

        print("1/2: Generando mapas de potencial de transición...")
        for r in range(self.ns, rows - self.ns):
            for c in range(self.ns, cols - self.ns):
                input_vector = self._get_prediction_input_vector(r, c)
                if input_vector is not None:
                    output_vector = self.mlp.propagate_forward(input_vector)
                    for i, cat in enumerate(self.crosstable.graduation_y):
                        potential_maps[cat][r, c] = output_vector[i]

        new_state_map = self.initial_raster.getBand(1).copy()
        categories = self.crosstable.graduation_x

        mascara_datos_validos = ~self.initial_raster.getBand(1).mask
        iteraciones_erosion = 3 # Cambia este número para 5x5 (2), 7x7 (3), etc.
        mascara_interior = ndi.binary_erosion(mascara_datos_validos, iterations=iteraciones_erosion)

        print(f"2/2: Asignando cambios (ignorando un borde de {iteraciones_erosion*2+1}x{iteraciones_erosion*2+1} píxeles)...")
        for initClass in categories:
            for finalClass in categories:
                if initClass == finalClass: continue
                demand = self.crosstable.getTransition(initClass, finalClass)
                if demand == 0: continue

                candidate_pixels_mask = (self.initial_raster.getBand(1).data == initClass)
                suitability = potential_maps[finalClass] * candidate_pixels_mask

                suitability[~mascara_interior] = -np.inf
                suitability[new_state_map != initClass] = -np.inf

                num_candidates = np.sum(suitability > -np.inf)
                if num_candidates < demand: demand = num_candidates

                if demand > 0:
                    flat_indices = np.argsort(suitability.flatten())[-demand:]
                    rows_cols = [np.unravel_index(i, suitability.shape) for i in flat_indices]
                    for r_idx, c_idx in rows_cols:
                        new_state_map[r_idx, c_idx] = finalClass

        final_raster = Raster()
        final_raster.create([new_state_map], self.initial_raster.geodata)
        return final_raster, potential_maps

print("Clases principales cargadas y listas.")

In [ ]:
# Cell 3: Project Configuration

# --- FILE PATHS IN GOOGLE DRIVE ---
carpeta_base = '/content/drive/MyDrive/prediccion/'

rutas = {
    "anho_2004":   carpeta_base + '2004.tif',
    "anho_2014":   carpeta_base + '2014.tif',
    "anho_2024":   carpeta_base + '2024.tif',
    "salida":    carpeta_base + 'mapa_predicho_2024.tif',

    "factores": [
        carpeta_base + 'aspecto.tif',
        carpeta_base + 'dem.tif',
        carpeta_base + 'slope.tif',
        carpeta_base + 'red_hidrog.tif',
        carpeta_base + 'iiee_2004.tif',
        carpeta_base + 'atractivos_turisticos.tif',
        carpeta_base + 'vias.tif',
        carpeta_base + 'poblaciones_grandes.tif',
        carpeta_base + 'poblaciones_dispersas.tif',
    ]
}

# optimizado
parametros = {
    "vecindad": 1,
    "muestras": 3000,
    #"modo_muestreo": "Random",
    "modo_muestreo": "Stratified",
    "porcentaje_validacion": 30,
    "epocas": 1000,
    "tasa_aprendizaje": 0.1,
    "momentum": 0.001,
    "capas_ocultas": [70, 30]
}

print("Configuración cargada.")


In [ ]:
# Cell 4: Data Loading and Verification
import os
import matplotlib.pyplot as plt

try:
    # --- DATA LOADING ---
    print("--- PASO 0: Cargando datos TIF ---")
    anho_2004 = Raster(rutas["anho_2004"])
    anho_2014 = Raster(rutas["anho_2014"])
    anho_2024 = Raster(rutas["anho_2024"])
    factor_rasters = [Raster(path) for path in rutas["factores"]]
    print("Datos cargados exitosamente.")
    print("-" * 60)

    # --- GEOMETRY VERIFICATION ---
    print("--- VERIFICANDO LA COINCIDENCIA DE GEOMETRÍA DE LOS RÁSTERS ---")
    verificacion_ok = True

    # 1. Check Initial vs. Final
    print("Comparando Mapa Inicial vs. Mapa Final...", end="")
    if anho_2004.geoDataMatch(anho_2014):
        print(" OK 2014 ")
    else:
        print(" FALLÓ ")
        verificacion_ok = False

    if anho_2004.geoDataMatch(anho_2024):
        print(" OK 2024 ")
    else:
        print(" FALLÓ ")
        verificacion_ok = False

    # 2. Check Initial vs. Each Factor
    for i, factor in enumerate(factor_rasters):
        nombre_factor = os.path.basename(rutas["factores"][i])
        print(f"Comparando Mapa Inicial vs. Factor '{nombre_factor}'...", end="")
        if anho_2004.geoDataMatch(factor):
            print(" OK ")
        else:
            print(" FALLÓ ")
            verificacion_ok = False

    print("-" * 60)
    if verificacion_ok:
        print("¡Éxito! Todas las geometrías de los rásters coinciden. Puedes continuar.")
    else:
        raise Exception("¡DETENIDO! Al menos un ráster no coincide. Corrige los archivos antes de continuar.")

except Exception as e:
    print(f"\n!!!!!! OCURRIÓ UN ERROR DURANTE LA CARGA O VERIFICACIÓN !!!!!!")
    print(str(e))

In [ ]:
# Celda 5 (MODIFIED for higher efficiency)

import pandas as pd
from sklearn.metrics import accuracy_score
import time
import copy
import datetime
import numpy as np # Asegurarse que numpy está importado

try:
    # --- PASO 1: MARKOV ANALYSIS (executed once) ---
    print("--- PASO 1: Calculando Matriz de Transición ---")
    crosstable = CrossTable(anho_2004.getBand(1), anho_2014.getBand(1))
    print("Matriz calculada.")
    print("-" * 60)

    # --- START OF HYPERPARAMETER SEARCH ---
    print("--- INICIANDO BÚSQUEDA DE HIPERPARÁMETROS CON REPETICIONES ---")

    numero_de_repeticiones = 5
    print(f"Cada combinación de parámetros se ejecutará {numero_de_repeticiones} veces para promediar resultados.")

    # 1. DEFINE SEARCH GRID
    # <-- CHANGE: Completed learning rate and momentum sequences
    param_grid = {
        "vecindad": [1],
        "muestras": [1000, 3000, 5000, 7000, 9000, 11000, 13000],
        "epocas": [5000, 7000, 9000, 11000],
        "tasa_aprendizaje": np.round(np.arange(0.0001, 0.001, 0.0002), 4).tolist(),
        "momentum": np.round(np.arange(0.001, 0.01, 0.002), 3).tolist(),
        "capas_ocultas": [[10, 5], [100, 50], [50, 50], [70,30], [100,100]]
    }

    # Variables para guardar el mejor resultado
    mejor_accuracy_promedio = -1.0
    resultados_historicos = []

    total_runs = len(param_grid["vecindad"]) * len(param_grid["muestras"]) * len(param_grid["epocas"]) * len(param_grid["tasa_aprendizaje"]) * len(param_grid["momentum"]) * len(param_grid["capas_ocultas"])
    run_count = 1

    print(f"Se probarán un total de {total_runs} combinaciones.")
    print("Normalizando factores de entrada (se realiza una sola vez)...")
    for f in factor_rasters: f.normalize(mode='mean')

    # BUCLES DE HIPERPARÁMETROS
    for vec in param_grid["vecindad"]:
        for mues in param_grid["muestras"]:
            # <-- CHANGE: Se eliminan impresiones de esta sección para reducir ruido
            # Generación de datos
            sampler = Sampler(anho_2004, factor_rasters, ns=vec)
            sampler.setTrainingData(anho_2014, mode=parametros["modo_muestreo"], samples=mues)
            porcentaje_val = parametros.get("porcentaje_validacion", 30)

            if len(sampler.data) < 4:
                print(f"\nADVERTENCIA: No se pudieron crear suficientes muestras ({len(sampler.data)}) para Muestras={mues}. Saltando...")
                run_count += (len(param_grid["capas_ocultas"]) * len(param_grid["epocas"]) * len(param_grid["tasa_aprendizaje"]) * len(param_grid["momentum"]))
                continue

            indice_corte = int(len(sampler.data) * (1 - porcentaje_val / 100))
            training_data = sampler.data[:indice_corte]
            validation_data = sampler.data[indice_corte:]

            if not training_data or not validation_data:
                print(f"\nADVERTENCIA: No se pudieron dividir las muestras para Muestras={mues}. Saltando...")
                run_count += (len(param_grid["capas_ocultas"]) * len(param_grid["epocas"]) * len(param_grid["tasa_aprendizaje"]) * len(param_grid["momentum"]))
                continue

            # Cálculo de neuronas
            neighborhood_size = (2 * vec + 1)**2
            num_factor_neurons = sum([f.bandcount * neighborhood_size for f in factor_rasters])
            num_state_neurons = (len(crosstable.graduation_x) - 1) * neighborhood_size
            input_neurons, output_neurons = num_factor_neurons + num_state_neurons, len(crosstable.graduation_y)
            state_categories = crosstable.graduation_x

            # Funciones de ayuda
            def get_input_vector(sample):
                dummy_vector = np.array([np.eye(len(state_categories))[state_categories.index(p)][:-1] if p != state_categories[-1] else np.zeros(len(state_categories) - 1) for p in sample['state']]).flatten()
                return np.hstack([dummy_vector, sample['factors']])
            def get_target_vector(sample):
                target = np.ones(output_neurons) * -1.0
                target[crosstable.graduation_y.index(sample['output'])] = 1.0
                return target

            for capas in param_grid["capas_ocultas"]:
                for epocas in param_grid["epocas"]:
                    for tasa in param_grid["tasa_aprendizaje"]:
                        for mom in param_grid["momentum"]:
                            # <-- CHANGE: Reemplaza múltiples prints por una sola línea de progreso
                            print(f"\rProbando [{run_count}/{total_runs}] | Mejor Acc: {mejor_accuracy_promedio:.2%}", end="")

                            accuracies_de_repeticion = []
                            for rep in range(numero_de_repeticiones):
                                arquitectura = [input_neurons] + capas + [output_neurons]
                                mlp = MLP(*arquitectura)
                                for i in range(epocas):
                                    sample = training_data[np.random.randint(len(training_data))]
                                    mlp.propagate_forward(get_input_vector(sample))
                                    mlp.propagate_backward(get_target_vector(sample), lrate=tasa, momentum=mom)

                                predicciones_val = [crosstable.graduation_y[np.argmax(mlp.propagate_forward(get_input_vector(s)))] for s in validation_data]
                                reales_val = [s['output'] for s in validation_data]
                                accuracies_de_repeticion.append(accuracy_score(reales_val, predicciones_val))

                            accuracy_promedio = np.mean(accuracies_de_repeticion)
                            accuracy_std = np.std(accuracies_de_repeticion)

                            resultados_historicos.append({
                                "vecindad": vec, "muestras": mues, "epocas": epocas,
                                "capas_ocultas": str(capas), "tasa_aprendizaje": tasa,
                                "momentum": mom, "accuracy_mean": accuracy_promedio,
                                "accuracy_std": accuracy_std
                            })

                            if accuracy_promedio > mejor_accuracy_promedio:
                                # <-- CHANGE: Única impresión importante durante la ejecución
                                print(f"\r Nuevo mejor promedio! [{run_count}/{total_runs}] Acc: {accuracy_promedio:.2%} (std:{accuracy_std:.2f}) con muestras={mues}, epocas={epocas}, capas={capas}, tasa={tasa}, mom={mom}")
                                mejor_accuracy_promedio = accuracy_promedio

                            run_count += 1

    # --- FINALIZACIÓN Y GUARDADO ---
    print("\n\n" + "="*60)
    print("--- BÚSQUEDA DE HIPERPARÁMETROS FINALIZADA ---")
    print("="*60)

    resultados_df = pd.DataFrame(resultados_historicos)
    resultados_df_ordenado = resultados_df.sort_values(by='accuracy_mean', ascending=False)

    # <-- CHANGE: Mostrar solo las 10 mejores combinaciones
    print(" Top 10 Mejores Combinaciones Encontradas ")
    print(resultados_df_ordenado.head(10).to_string(index=False))
    print("\n" + "-"*60)

    print("Guardando tabla completa de resultados en un archivo CSV...")
    timestamp = datetime.datetime.now().strftime("%Y_%m_%d_%H_%M")
    nombre_archivo_csv = f'resultados_hiperparametros_PROMEDIO_{timestamp}.csv'
    ruta_csv = carpeta_base + nombre_archivo_csv
    resultados_df_ordenado.to_csv(ruta_csv, index=False)
    print(f" ¡Resultados guardados exitosamente en: {ruta_csv}!")

except Exception as e:
    print(f"\n!!!!!! OCURRIÓ UN ERROR !!!!!!")
    import traceback
    traceback.print_exc()